In [4]:
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

import matplotlib.pyplot as plt
import numpy as np

## Impurity: Gini and Entropy

In [2]:
def gini(y):
    if len(y) == 0:
        return 0.0
    
    p = np.bincount(y) / len(y)
    return 1 - np.sum(p ** 2)

def entropy(y):
    if len(y) == 0:
        return 0.0
    
    counts = np.bincount(y)
    p = counts[counts > 0] / len(y)
    return -np.sum(p * np.log2(p))


def impurity(y, criterion = 'gini'):
    if criterion == 'gini':
        return gini(y)
    else:
        return entropy(y)

## Information Gain

In [7]:
def information_gain(y, y_left, y_right, criterion = 'gini'):
    n = len(y)
    n_left = len(y_left)
    n_right = len(y_right)
    
    parent_imp = impurity(y, criterion)
    child_imp = (n_left / n) * impurity(y_left, criterion) + (n_right / n) * impurity(y_right, criterion)
    return parent_imp - child_imp

In [8]:
def best_split(X, y, criterion = 'gini'):
    
    n_features = X.shape[1]
    best_gain, best_feature, best_threshold = -1.0, None, None

    for feature_idx in range(n_features):
        values = np.unique(X[:, feature_idx])
        thresholds = (values[:-1] + values[1:]) / 2

        for t in thresholds:
            mask = X[:, feature_idx] <= t
            y_left, y_right = y[mask], y[~mask]
            if len(y_left) == 0 or len(y_right) == 0:
                continue
            gain = information_gain(y, y_left, y_right, criterion)
            if gain > best_gain:
                best_gain, best_feature, best_threshold = gain, feature_idx, t

    return best_feature, best_threshold, best_gain

In [ ]:
class Node:
    def __init__(self, leaf = False, prediction = None, feature = None, threshold = None, 
                 left = None, right = None):
        self.leaf = leaf
        self.prediction = prediction
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right


def grow_tree(X, y, depth = 0, max_depth = 3, min_samples_leaf = 1, criterion = 'gini'):
    majority_class = np.bincount(y).argmax()

    if depth >= max_depth or len(np.unique(y)) == 1 or len(y) < 2 * min_samples_leaf:
        return Node(leaf=True, prediction=majority_class)

    feature, threshold, gain = best_split(X, y, criterion)
    if feature is None or gain <= 0:
        return Node(leaf=True, prediction=majority_class)

    mask = X[:, feature] <= threshold
    left = grow_tree(X[mask], y[mask], depth + 1, max_depth, min_samples_leaf, criterion)
    right = grow_tree(X[~mask], y[~mask], depth + 1, max_depth, min_samples_leaf, criterion)
    return Node(leaf=False, feature=feature, threshold=threshold, left=left, right=right)


def predict_one(node, x):
    while not node.leaf:
        node = node.left if x[node.feature] <= node.threshold else node.right
    return node.prediction


def predict(tree, X):
    return np.array([predict_one(tree, x) for x in X])